<a href="https://colab.research.google.com/github/OdysseusPolymetis/initiation_ia/blob/main/GANS_1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# StyleGAN2 Image Generation

# StyleGAN2 : générer des visages synthétiques

Dans ce notebook, on utilise un modèle appelé **StyleGAN2** pour produire des images de visages qui n'existent pas.

## Idée principale
Le modèle a déjà été entraîné auparavant sur un très grand nombre d'images.  
Ici, on **ne l'entraîne pas** : on s'en sert uniquement pour **générer** de nouvelles images.

## Ce qu'on veut comprendre
On va voir trois choses :

1. le rôle du **modèle préentraîné** ;
2. le rôle du **seed** ;
3. le rôle de la **troncature**.

## À retenir
Dans ce notebook, la logique générale est :

**modèle + seed + paramètres -> image générée**

## Installation des dépendances

On installe :
- PyTorch ;
- quelques bibliothèques Python utiles ;
- `ninja-build`, utilisé par certains composants du dépôt NVIDIA.

In [ ]:
!nvidia-smi || true

import sys
import platform

print("Python :", sys.version)
print("Plateforme :", platform.platform())

In [ ]:
!apt-get -qq update
!apt-get -qq install -y build-essential ninja-build

!pip -q install numpy pillow matplotlib

In [ ]:
import torch
print("Torch version :", torch.__version__)
print("CUDA disponible :", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU :", torch.cuda.get_device_name(0))

## Clonage du dépôt officiel StyleGAN2-ADA

On récupère ici le code officiel de NVIDIA.
C'est ce dépôt qui contient le script `generate.py`.

In [ ]:
!rm -rf /content/stylegan2-ada-pytorch
!git clone https://github.com/NVlabs/stylegan2-ada-pytorch.git /content/stylegan2-ada-pytorch
%cd /content/stylegan2-ada-pytorch

## Choisir un modèle préentraîné

Un **modèle préentraîné** est un réseau qui a déjà appris à produire un certain type d'images.

Ici, on utilise deux exemples :

- `ffhq` : visages photoréalistes ;
- `metfaces` : portraits de style artistique.

### Pourquoi cela change-t-il les images ?
Parce que le modèle n'a pas appris sur les mêmes types d'images.

- un modèle entraîné sur des visages réalistes génère des visages réalistes ;
- un modèle entraîné sur des portraits peints génère des portraits peints.

### À retenir
Le modèle choisi détermine **le type d'univers visuel** dans lequel on génère.

In [ ]:
PRETRAINED_MODELS = {
    "ffhq": "https://nvlabs-fi-cdn.nvidia.com/stylegan2-ada-pytorch/pretrained/ffhq.pkl",
    "metfaces": "https://nvlabs-fi-cdn.nvidia.com/stylegan2-ada-pytorch/pretrained/metfaces.pkl",
}

model_name = "ffhq"
network_url = PRETRAINED_MODELS[model_name]

print("Modèle choisi :", model_name)
print("URL du modèle :", network_url)

## Quelles instructions donne-t-on au modèle ?

Dans StyleGAN2, on ne donne pas une phrase comme dans un modèle à prompt.

On donne surtout :
- un **modèle préentraîné** ;
- un **seed** ;
- quelques **paramètres** comme la troncature.

### Ce qui se passe ensuite
Le seed sert à produire un vecteur de nombres aléatoires.  
Ce vecteur est ensuite transformé par le générateur en image.

### Important
L'image n'est pas stockée à l'avance.  
Elle est calculée au moment de l'exécution à partir du modèle déjà entraîné.

## Qu'est-ce qu'un espace latent ?

Le modèle ne reçoit pas directement une image ou une phrase.  
Il reçoit un **vecteur latent**, c'est-à-dire une liste de nombres.

### Vecteur latent
Un vecteur latent est une représentation numérique interne d'une image possible.

### Espace latent
L'espace latent est l'ensemble de tous les vecteurs possibles que le modèle peut transformer en images.

On peut l'imaginer comme une grande carte abstraite :
- un point de cette carte correspond à une image possible ;
- un autre point correspond à une autre image ;
- des points proches donnent souvent des images assez proches.

### Lien avec le seed
Le **seed** sert à générer un vecteur latent de départ.

On peut donc résumer ainsi :

**seed -> vecteur latent -> image**

## Pourquoi est-ce utile ?

Cette idée est importante, car elle permet de comprendre que le modèle :

- ne pioche pas simplement une image existante ;
- ne suit pas une phrase comme un prompt ;
- construit une image à partir d'une représentation numérique apprise.

C'est pour cela qu'on peut ensuite :
- générer plusieurs visages différents ;
- interpoler entre deux visages ;
- modifier certaines propriétés de manière progressive.

## Première génération : une seule image

Ici, on génère une image avec :
- le modèle `ffhq`
- le `seed` 42
- une troncature à `0.7`

Le **seed** fixe le point de départ aléatoire.  
Si on garde le même seed et le même modèle, on retrouve la même image.

In [ ]:
!python generate.py \
    --outdir=/content/output_single \
    --trunc=0.7 \
    --seeds=42 \
    --network="$network_url"

In [ ]:
import os
from PIL import Image
import matplotlib.pyplot as plt

img_path = "/content/output_single/seed0042.png"
assert os.path.exists(img_path), f"Image introuvable : {img_path}"

img = Image.open(img_path)

plt.figure(figsize=(6, 6))
plt.imshow(img)
plt.title("Image générée avec seed 42")
plt.axis("off")
plt.show()

## Générer plusieurs images

On garde le même modèle, mais on change les seeds.

Cela permet de montrer que le modèle ne produit pas une seule image,
mais une **famille d'images plausibles**.

In [ ]:
seeds = "1,2,3,4,5,6"

!python generate.py \
    --outdir=/content/output_many \
    --trunc=0.7 \
    --seeds="$seeds" \
    --network="$network_url"

In [ ]:
import glob

files = sorted(glob.glob("/content/output_many/*.png"))
print("Images générées :")
for f in files:
    print("-", os.path.basename(f))

plt.figure(figsize=(14, 8))
for i, fp in enumerate(files, start=1):
    plt.subplot(2, 3, i)
    plt.imshow(Image.open(fp))
    plt.title(os.path.basename(fp))
    plt.axis("off")

plt.tight_layout()
plt.show()

## Comprendre la troncature

La **troncature** est un paramètre qui influence la manière dont le modèle explore les images possibles.

### Idée simple
Le modèle connaît beaucoup d'images possibles.  
La troncature règle à quel point on reste dans des zones très typiques de ce qu'il a appris.

### Intuition
- troncature plus faible : images souvent plus "moyennes", plus sages, plus régulières ;
- troncature plus élevée : images souvent plus variées, parfois plus originales, parfois plus étranges.

### Important
La troncature ne change pas le modèle lui-même.  
Elle change seulement la manière d'échantillonner dans l'espace latent.

### Valeurs typiques
- `trunc = 1.0` : plus de liberté
- `trunc = 0.7` : bon compromis
- `trunc = 0.3` : plus centré, plus typique

## Effet de la troncature

La troncature contrôle en partie le compromis entre :
- diversité ;
- régularité visuelle.

On va comparer ici trois valeurs :
- `1.0`
- `0.7`
- `0.3`

On garde le **même seed** pour que seule la troncature change.

In [ ]:
trunc_values = [1.0, 0.7, 0.3]
seed = 42

for t in trunc_values:
    outdir = f"/content/trunc_{str(t).replace('.', '_')}"
    !python generate.py \
        --outdir="$outdir" \
        --trunc={t} \
        --seeds={seed} \
        --network="$network_url"

In [ ]:
paths = [
    "/content/trunc_1_0/seed0042.png",
    "/content/trunc_0_7/seed0042.png",
    "/content/trunc_0_3/seed0042.png",
]
titles = ["trunc = 1.0", "trunc = 0.7", "trunc = 0.3"]

plt.figure(figsize=(15, 5))

for i, (path, title) in enumerate(zip(paths, titles), start=1):
    plt.subplot(1, 3, i)
    plt.imshow(Image.open(path))
    plt.title(title)
    plt.axis("off")

plt.tight_layout()
plt.show()

## Changer de domaine visuel : MetFaces

On change maintenant de modèle.

Au lieu de produire des visages photoréalistes comme `ffhq`,
on utilise `metfaces`, un modèle entraîné sur des portraits artistiques.

On garde le même principe :
- un modèle ;
- un seed ;
- une image générée.

In [ ]:
network_url = PRETRAINED_MODELS["metfaces"]
print("Nouveau modèle :", network_url)

!python generate.py \
    --outdir=/content/output_metfaces \
    --trunc=0.7 \
    --seeds=42 \
    --network="$network_url"

In [ ]:
img = Image.open("/content/output_metfaces/seed0042.png")

plt.figure(figsize=(6, 6))
plt.imshow(img)
plt.title("MetFaces — seed 42")
plt.axis("off")
plt.show()

## Questions d'observation

1. Pourquoi deux seeds différents donnent-ils deux visages différents ?
2. Pourquoi le même seed ne donne-t-il pas la même image si l'on change de modèle ?
3. Que change la troncature dans l'apparence de l'image ?
4. En quoi cette génération diffère-t-elle d'un face swap ?
5. Peut-on dire que le modèle "copie" une image existante ?

## Ce qu'il faut retenir

Dans ce notebook, on a vu que :

- un GAN préentraîné peut générer une image sans image d'entrée ;
- le **seed** contrôle le tirage aléatoire ;
- le **modèle choisi** change le type d'image produit ;
- la **troncature** influence le rendu final.

La logique générale est donc :

**modèle préentraîné + seed -> image synthétique**